In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Libraries loaded successfully")

In [ ]:

df_2009 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2009-2010')
df_2010 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2010-2011')

# Combine both years
df = pd.concat([df_2009, df_2010], ignore_index=True)

print(f"Shape: {df.shape}")
print(f"Total records: {len(df):,}")
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

In [ ]:

print("=" * 50)
print("DATASET INFO")
print("=" * 50)
df.info()

print("\n" + "=" * 50)
print("STATISTICAL SUMMARY")
print("=" * 50)
df.describe()

In [ ]:
# Missing value analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print("MISSING VALUES REPORT")
print(missing_df[missing_df['Missing Count'] > 0])

# Visualize
plt.figure(figsize=(10, 4))
missing_pct[missing_pct > 0].plot(kind='bar', color='steelblue')
plt.title('Missing Values by Column (%)')
plt.ylabel('Missing %')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../results/cluster_plots/missing_values.png', dpi=150)
plt.show()
print("✅ Plot saved")

In [ ]:
# Handle missing values

print(f"Records before cleaning: {len(df):,}")

# 1. Drop rows with no CustomerID — we can't do customer segmentation without it
df.dropna(subset=['Customer ID'], inplace=True)
print(f"After dropping missing CustomerID: {len(df):,}")

# 2. Drop rows with missing Description (very few, negligible)
df.dropna(subset=['Description'], inplace=True)
print(f"After dropping missing Description: {len(df):,}")

# Confirm no more missing values
print("\nMissing values remaining:")
print(df.isnull().sum())

In [ ]:
# Remove cancellations and invalid records

# Cancellations have InvoiceNo starting with 'C'
df['is_cancelled'] = df['Invoice'].astype(str).str.startswith('C')
cancelled_count = df['is_cancelled'].sum()
print(f"Cancelled transactions: {cancelled_count:,}")

# Keep a record of cancellation rate per customer (useful feature later)
cancellation_rate = df.groupby('Customer ID')['is_cancelled'].mean().reset_index()
cancellation_rate.columns = ['Customer ID', 'cancellation_rate']

# Remove cancelled orders from main df
df = df[~df['is_cancelled']].copy()
print(f"After removing cancellations: {len(df):,}")

# Remove rows where Quantity <= 0
df = df[df['Quantity'] > 0]
print(f"After removing non-positive quantities: {len(df):,}")

# Remove rows where Price <= 0
df = df[df['Price'] > 0]
print(f"After removing non-positive prices: {len(df):,}")

df.drop(columns=['is_cancelled'], inplace=True)

In [ ]:
# Create core features needed for segmentation

# Total spend per transaction
df['TotalSpend'] = df['Quantity'] * df['Price']

# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Reference date for Recency calculation (day after last transaction)
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"Reference date for Recency: {reference_date.date()}")

print("\nSample with TotalSpend:")
df[['Customer ID', 'Invoice', 'Quantity', 'Price', 'TotalSpend', 'InvoiceDate']].head()

In [ ]:
# Build RFM + extended features at customer level

customer_df = df.groupby('Customer ID').agg(
    # RFM core
    Recency        = ('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency      = ('Invoice',     'nunique'),
    Monetary       = ('TotalSpend',  'sum'),

    # Behavioral features
    AvgOrderValue  = ('TotalSpend',  'mean'),
    TotalItems     = ('Quantity',    'sum'),
    AvgItemsPerOrder = ('Quantity',  'mean'),
    UniqueProducts = ('StockCode',   'nunique'),
    NumCountries   = ('Country',     'nunique'),
    FirstPurchase  = ('InvoiceDate', 'min'),
    LastPurchase   = ('InvoiceDate', 'max'),
).reset_index()

# Derived features
customer_df['CustomerAge'] = (reference_date - customer_df['FirstPurchase']).dt.days
customer_df['AvgDaysBetweenOrders'] = customer_df['CustomerAge'] / customer_df['Frequency']
customer_df['SpendPerItem'] = customer_df['Monetary'] / customer_df['TotalItems']

# Merge cancellation rate
customer_df = customer_df.merge(cancellation_rate, on='Customer ID', how='left')
customer_df['cancellation_rate'] = customer_df['cancellation_rate'].fillna(0)

# Drop date columns (not needed for clustering)
customer_df.drop(columns=['FirstPurchase', 'LastPurchase'], inplace=True)

print(f"Customer-level dataset shape: {customer_df.shape}")
print(f"Total unique customers: {len(customer_df):,}")
print(f"\nFeatures created: {customer_df.columns.tolist()}")
customer_df.head()

In [ ]:
# Outlier detection and capping

features_to_check = ['Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 'TotalItems']

fig, axes = plt.subplots(1, len(features_to_check), figsize=(18, 4))
for i, col in enumerate(features_to_check):
    axes[i].boxplot(customer_df[col].dropna())
    axes[i].set_title(col)
    axes[i].set_xlabel('')
plt.suptitle('Outlier Detection — Before Capping', fontsize=13)
plt.tight_layout()
plt.savefig('../results/cluster_plots/outliers_before.png', dpi=150)
plt.show()

# Cap outliers at 99th percentile (Winsorization)
for col in features_to_check:
    upper = customer_df[col].quantile(0.99)
    customer_df[col] = customer_df[col].clip(upper=upper)
    print(f"{col}: capped at {upper:.2f}")

print("\n✅ Outliers capped at 99th percentile")

In [ ]:
# Save the processed customer-level dataset

customer_df.to_csv('../data/processed/customer_features.csv', index=False)

print("✅ Saved to: data/processed/customer_features.csv")
print(f"\nFinal dataset shape: {customer_df.shape}")
print(f"\nFeature list ({len(customer_df.columns)} columns):")
for i, col in enumerate(customer_df.columns, 1):
    print(f"  {i:2}. {col}")

In [ ]:
# Summary

print("=" * 55)
print("  PREPROCESSING COMPLETE — SUMMARY")
print("=" * 55)
print(f"  Raw transactions loaded    : {len(df_2009) + len(df_2010):>10,}")
print(f"  After removing no-CustomerID: {len(df):>10,}")
print(f"  Unique customers segmented : {len(customer_df):>10,}")
print(f"  Features engineered        : {len(customer_df.columns) - 1:>10}")
print(f"  RFM features               :          3")
print(f"  Behavioral features        :          9")
print(f"  Outlier method             :   Winsorize (99th pct)")
print(f"  Saved to                   : data/processed/")
print("=" * 55)
print("\n✅ Ready for EDA (Notebook 02)")